# Prompt Style Benchmark — Trigger Words vs Verbose

Tests whether short trigger-word prompts produce equivalent quality to verbose prompts, or whether verbose prompts are necessary for richer output.

**5 styles tested:**
1. Trigger words only (~15 tokens)
2. Short imperative (~25 tokens)
3. Standard German (~40 tokens) — current baseline
4. Detailed structured (~80 tokens)
5. Verbose with examples (~200 tokens)

Each tested 3 times on SA5 architecture (gpt-4o-mini single-shot).

**Cost:** ~$0.10 | **Runtime:** ~3-4 minutes

## Cell 1 — Setup

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# Tests how prompt style (trigger words → verbose) affects latency + quality
# ═══════════════════════════════════════════════════════════════════
import sys, os, time, json, copy, csv, traceback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

CANDIDATES = [
    "/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch",
    os.path.abspath(".."), os.getcwd(),
]
PROJECT_ROOT = next(
    (p for p in CANDIDATES if os.path.isfile(os.path.join(p, "generate_visualization.py"))), None
)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from retrieve_data import retrieve_data
from generate_visualization import Agent
from init_phoenix import init_phoenix
from visualization_from_template import generate_from_template

import phoenix as px
if not hasattr(px, "_thesis_initialised"):
    client, tool_client, tracer = init_phoenix("rq3-prompt-style")
    px._thesis_initialised = True
    px._client = client; px._tool = tool_client; px._tracer = tracer
else:
    client = px._client; tool_client = px._tool; tracer = px._tracer

MD_TABLE = retrieve_data(None, type="test")
agent = Agent(client, tool_client, tracer, "o4-mini")

os.makedirs("prompt_style_results", exist_ok=True)
os.makedirs("prompt_style_results/renders", exist_ok=True)

print("Setup OK — SA5 architecture (single-shot extract, gpt-4o-mini)")


Setup OK — SA5 architecture (single-shot extract, gpt-4o-mini)


## Cell 2 — The 5 prompt styles

In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Five prompt styles, same intent
# ═══════════════════════════════════════════════════════════════════

PROMPT_STYLES = [
    {
        "id": "S1_trigger_words",
        "name": "Trigger words only",
        "prompt": "Teckentrup JVA 2021-2024 Umsatz monatlich",
        "description": "Just keywords, no sentence structure",
    },
    {
        "id": "S2_short_imperative",
        "name": "Short imperative",
        "prompt": "Show monthly Teckentrup JVA revenue 2021-2024 with annotations.",
        "description": "Direct command, English, brief",
    },
    {
        "id": "S3_standard_german",
        "name": "Standard (current baseline)",
        "prompt": ("Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 "
                   "im Segment JVA? Provide a detailed analysis and a comprehensive visualization."),
        "description": "Current default question, mixed German/English",
    },
    {
        "id": "S4_detailed",
        "name": "Detailed structured",
        "prompt": ("Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? "
                   "Provide a detailed monthly breakdown analysis with year-over-year comparisons. "
                   "Identify peaks, troughs, and seasonal patterns. Annotate notable anomalies. "
                   "Use clear axis labels in EUR and German month names."),
        "description": "Standard + structure + style guidance",
    },
    {
        "id": "S5_verbose",
        "name": "Verbose with examples",
        "prompt": ("I need a comprehensive multi-dimensional analysis of Teckentrup’s revenue "
                   "performance in the JVA (Justizvollzugsanstalten / correctional facilities) segment "
                   "from 2021 through 2024. Include: monthly revenue trends, year-over-year growth rates, "
                   "seasonal patterns, peak periods (e.g. Jun 2022 spike, Nov 2022 second peak), "
                   "low-performance periods (e.g. Sep 2024 zero), cumulative annual totals. "
                   "Format as a multi-line chart with German axis labels (Monat / Umsatz in EUR), "
                   "annotated callouts for the top 2 highest months and the lowest, "
                   "legend showing 2021/2022/2023/2024 as separate series with distinct colors."),
        "description": "Standard + examples + explicit chart specifications",
    },
]

# Approximate token counts
def approx_tokens(text):
    return len(text) // 4

print(f"{'Style':<25} {'~Tokens':>8} {'Description':<50}")
print("-" * 95)
for s in PROMPT_STYLES:
    s["approx_tokens"] = approx_tokens(s["prompt"])
    print(f"{s['name']:<25} {s['approx_tokens']:>8} {s['description']:<50}")


Style                      ~Tokens Description                                       
-----------------------------------------------------------------------------------------------
Trigger words only              10 Just keywords, no sentence structure              
Short imperative                15 Direct command, English, brief                    
Standard (current baseline)       34 Current default question, mixed German/English    
Detailed structured             70 Standard + structure + style guidance             
Verbose with examples          152 Standard + examples + explicit chart specifications


## Cell 3 — Quality validator

In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Quality validator (same 8-check as architecture benchmark)
# ═══════════════════════════════════════════════════════════════════
VALID_CT = {"line","bar","pie","scatter","column","stackedbar"}

def validate(cfg, render=True, tag="?", run=0):
    r = {"schema_complete":False,"valid_charttype":False,"has_data":False,
         "has_title":False,"has_axes":False,"renderable":False,
         "has_annotations":False,"data_consistency":False,
         "quality_score":0,"passed":False,"error":None,
         "n_data_groups":0,"n_annotations":0,"title_text":""}
    
    if cfg is None: r["error"]="null"; return r
    if hasattr(cfg, "model_dump"): cfg = cfg.model_dump()
    if not isinstance(cfg, dict): r["error"]="not dict"; return r
    
    req = ["titlename","charttype","xlabel","ylabel","data"]
    r["schema_complete"] = all(cfg.get(f) for f in req)
    
    ct = cfg.get("charttype","")
    if hasattr(ct,"value"): ct=ct.value
    r["valid_charttype"] = str(ct).lower().strip() in VALID_CT
    
    data = cfg.get("data") or []
    r["has_data"] = len(data) > 0
    r["n_data_groups"] = len(data)
    r["has_title"] = bool(str(cfg.get("titlename","")).strip())
    r["title_text"] = str(cfg.get("titlename",""))[:60]
    r["has_axes"] = bool(cfg.get("xlabel")) and bool(cfg.get("ylabel"))
    
    consistent = True
    for g in data:
        d = g if isinstance(g,dict) else (g.model_dump() if hasattr(g,"model_dump") else {})
        xd, yd = d.get("x_data",[]) or [], d.get("y_data",[]) or []
        if not xd or not yd or len(xd) != len(yd): consistent=False; break
    r["data_consistency"] = consistent
    
    annos = cfg.get("annotations") or []
    r["has_annotations"] = len(annos) > 0
    r["n_annotations"] = len(annos)
    
    if render and r["has_data"] and r["valid_charttype"]:
        try:
            plt.close("all")
            generate_from_template(copy.deepcopy(cfg))
            fn = f"prompt_style_results/renders/{tag}_run{run:02d}.png"
            plt.savefig(fn, dpi=100, bbox_inches="tight")
            plt.close("all")
            r["renderable"] = True
        except Exception as e:
            r["error"] = f"render: {str(e)[:60]}"
            plt.close("all")
    
    checks = ["schema_complete","valid_charttype","has_data","has_title",
              "has_axes","renderable","has_annotations","data_consistency"]
    r["quality_score"] = sum(r[c] for c in checks)
    r["passed"] = r["quality_score"] >= 5
    return r

print("Validator ready")


Validator ready


## Cell 4 — Run 5 styles × 3 runs

In [12]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Run benchmark: 5 styles × 3 runs each
# 15 total calls, ~3-4 minutes, ~$0.10
# ═══════════════════════════════════════════════════════════════════

N_RUNS = 3  # 3 runs per style for time budget

CSV_FILE = "prompt_style_results/style_benchmark.csv"
fields = ["style_id","style_name","approx_tokens","run","latency_s",
          "quality_score","passed","schema_complete","valid_charttype","has_data",
          "renderable","has_annotations","data_consistency","n_data_groups","n_annotations",
          "charttype","title_text","error"]

with open(CSV_FILE,"w",newline="") as f:
    csv.DictWriter(f, fieldnames=fields).writeheader()

print(f"Running {len(PROMPT_STYLES)} styles × {N_RUNS} runs = {len(PROMPT_STYLES)*N_RUNS} calls\n")

for s in PROMPT_STYLES:
    print(f"{'='*70}")
    print(f"  {s['id']}: {s['name']}  (~{s['approx_tokens']} tokens)")
    print(f"  prompt: {s['prompt'][:80]}{'...' if len(s['prompt'])>80 else ''}")
    print(f"{'='*70}")
    
    for i in range(N_RUNS):
        print(f"  Run {i+1}/{N_RUNS} ... ", end="", flush=True)
        t0 = time.perf_counter()
        try:
            raw = agent.extract_chart_config(MD_TABLE, analysis=s["prompt"])
            lat = round(time.perf_counter()-t0, 3)
            if hasattr(raw,"model_dump"): raw = raw.model_dump()
            if isinstance(raw,dict) and "content" in raw:
                c = raw["content"]
                raw = json.loads(c) if isinstance(c,str) else c
            qr = validate(raw, render=True, tag=s["id"], run=i+1)
            
            charttype = ""
            if isinstance(raw,dict):
                ct = raw.get("charttype","")
                if hasattr(ct,"value"): ct=ct.value
                charttype = str(ct)
            
            status = f"PASS {qr['quality_score']}/8" if qr["passed"] else f"FAIL {qr.get('error','?')[:40]}"
            print(f"{lat:6.2f}s  {status}  chart={charttype}  groups={qr['n_data_groups']}  annos={qr['n_annotations']}")
        except Exception as e:
            lat = round(time.perf_counter()-t0, 3)
            qr = {k:False for k in ["schema_complete","valid_charttype","has_data","renderable",
                                     "has_annotations","data_consistency","passed"]}
            qr.update({"quality_score":0,"n_data_groups":0,"n_annotations":0,
                       "error":str(e)[:80],"title_text":""})
            charttype = ""
            print(f"{lat:6.2f}s  CRASH {str(e)[:40]}")
        
        row = {"style_id": s["id"], "style_name": s["name"],
               "approx_tokens": s["approx_tokens"], "run": i+1, "latency_s": lat,
               "charttype": charttype,
               **{k: qr.get(k) for k in fields[5:-2] if k in qr},
               "title_text": qr.get("title_text",""),
               "error": qr.get("error","")}
        with open(CSV_FILE,"a",newline="") as f:
            csv.DictWriter(f, fieldnames=fields).writerow(row)

print(f"\nSaved: {CSV_FILE}")


Running 5 styles × 3 runs = 15 calls

  S1_trigger_words: Trigger words only  (~10 tokens)
  prompt: Teckentrup JVA 2021-2024 Umsatz monatlich
  Run 1/3 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 19.60s  PASS 8/8  chart=line  groups=4  annos=4
  Run 2/3 ...  14.47s  PASS 6/8  chart=line  groups=4  annos=0
  Run 3/3 ...  14.64s  PASS 8/8  chart=line  groups=4  annos=1
  S2_short_imperative: Short imperative  (~15 tokens)
  prompt: Show monthly Teckentrup JVA revenue 2021-2024 with annotations.
  Run 1/3 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 17.82s  PASS 8/8  chart=line  groups=4  annos=2
  Run 2/3 ... 

...

 41.01s  PASS 8/8  chart=line  groups=4  annos=4
  S5_verbose: Verbose with examples  (~152 tokens)
  prompt: I need a comprehensive multi-dimensional analysis of Teckentrup’s revenue perfor...
  Run 1/3 ...  39.35s  PASS 8/8  chart=line  groups=4  annos=3
  Run 2/3 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 34.11s  PASS 8/8  chart=line  groups=4  annos=3
  Run 3/3 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 45.82s  PASS 8/8  chart=line  groups=4  annos=3

Saved: prompt_style_results/style_benchmark.csv


## Cell 5 — Latency and quality analysis

In [13]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Analysis: latency + quality + visual richness per style
# ═══════════════════════════════════════════════════════════════════

df = pd.read_csv(CSV_FILE)
df["latency_s"]      = pd.to_numeric(df["latency_s"], errors="coerce")
df["quality_score"]  = pd.to_numeric(df["quality_score"], errors="coerce")
df["n_annotations"]  = pd.to_numeric(df["n_annotations"], errors="coerce")
df["n_data_groups"]  = pd.to_numeric(df["n_data_groups"], errors="coerce")
df["passed"]         = df["passed"].astype(str).isin(["True","1","1.0"])

summary = df.groupby("style_id").agg(
    style_name=("style_name","first"),
    tokens=("approx_tokens","first"),
    lat_mean=("latency_s","mean"),
    lat_std=("latency_s","std"),
    qual_mean=("quality_score","mean"),
    pass_rate=("passed","mean"),
    annos_mean=("n_annotations","mean"),
    groups_mean=("n_data_groups","mean"),
).reset_index()
summary["pass_rate"] = summary["pass_rate"] * 100

ORDER = ["S1_trigger_words","S2_short_imperative","S3_standard_german","S4_detailed","S5_verbose"]
summary = summary.set_index("style_id").reindex(ORDER).reset_index()

print(f"\n{'='*100}")
print("  PROMPT STYLE BENCHMARK SUMMARY")
print(f"{'='*100}")
print(f"  {'Style':<25} {'Tokens':>7} {'Latency':>14} {'Quality':>9} {'Pass%':>6} {'Annos':>6} {'Groups':>7}")
print("-"*100)

for _, r in summary.iterrows():
    print(f"  {r['style_name']:<25} {int(r['tokens']):>7} "
          f"{r['lat_mean']:>5.2f}s ± {r['lat_std']:>4.1f} "
          f"{r['qual_mean']:>6.2f}/8  {r['pass_rate']:>5.0f}%  "
          f"{r['annos_mean']:>5.1f}   {r['groups_mean']:>5.1f}")
print("="*100)

# Correlation
x = summary["tokens"].values
y_lat = summary["lat_mean"].values
y_qual = summary["qual_mean"].values

if len(x) >= 2:
    r_lat = np.corrcoef(x, y_lat)[0,1]
    r_qual = np.corrcoef(x, y_qual)[0,1]
    print(f"\n  Correlation (tokens, latency):   r = {r_lat:.3f}")
    print(f"  Correlation (tokens, quality):   r = {r_qual:.3f}")
    print(f"  Correlation (tokens, annotations): r = {np.corrcoef(x, summary['annos_mean'])[0,1]:.3f}")



  PROMPT STYLE BENCHMARK SUMMARY
  Style                      Tokens        Latency   Quality  Pass%  Annos  Groups
----------------------------------------------------------------------------------------------------
  Trigger words only             10 16.24s ±  2.9   7.33/8    100%    1.7     4.0
  Short imperative               15 21.91s ±  3.7   8.00/8    100%    2.0     4.0
  Standard (current baseline)      34 225.97s ± 347.1   8.00/8    100%    4.0     1.0
  Detailed structured            70 33.49s ± 10.3   8.00/8    100%    3.7     4.0
  Verbose with examples         152 39.76s ±  5.9   8.00/8    100%    3.0     4.0

  Correlation (tokens, latency):   r = -0.115
  Correlation (tokens, quality):   r = 0.441
  Correlation (tokens, annotations): r = 0.377


## Cell 6 — Figures (4-panel summary)

In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Figures
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top-left: latency per style
ax = axes[0,0]
colors = ["#1D9E75","#2E75B6","#E6A817","#C0392B","#9B59B6"]
bars = ax.bar(range(len(summary)), summary["lat_mean"], yerr=summary["lat_std"],
              color=colors, edgecolor="white", capsize=5)
for bar, v in zip(bars, summary["lat_mean"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f"{v:.1f}s", ha="center", fontweight="bold")
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary["style_name"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Latency (s)")
ax.set_title("Latency per Prompt Style", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# Top-right: quality per style
ax = axes[0,1]
bars = ax.bar(range(len(summary)), summary["qual_mean"],
              color=colors, edgecolor="white")
for bar, v in zip(bars, summary["qual_mean"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f"{v:.1f}", ha="center", fontweight="bold")
ax.axhline(5, color="red", linestyle="--", alpha=0.5, label="Pass threshold (5)")
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary["style_name"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Quality score (/8)")
ax.set_title("Quality per Prompt Style", fontweight="bold")
ax.set_ylim(0, 9); ax.legend()
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# Bottom-left: annotations richness per style
ax = axes[1,0]
bars = ax.bar(range(len(summary)), summary["annos_mean"],
              color=colors, edgecolor="white")
for bar, v in zip(bars, summary["annos_mean"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f"{v:.1f}", ha="center", fontweight="bold")
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary["style_name"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Mean annotations per chart")
ax.set_title("Visual Richness (Annotations)", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# Bottom-right: latency vs tokens
ax = axes[1,1]
ax.errorbar(summary["tokens"], summary["lat_mean"], yerr=summary["lat_std"],
            fmt="o", markersize=12, capsize=8, color="#2E75B6", ecolor="#2E75B6", linewidth=2)
if len(summary) >= 2:
    slope, intercept = np.polyfit(summary["tokens"], summary["lat_mean"], 1)
    xline = np.linspace(summary["tokens"].min(), summary["tokens"].max(), 100)
    ax.plot(xline, slope*xline+intercept, "--", color="#C0392B",
            label=f"Slope: {slope*1000:.1f}s/1k tokens")
    r = np.corrcoef(summary["tokens"], summary["lat_mean"])[0,1]
    ax.text(0.05, 0.95, f"r = {r:.3f}", transform=ax.transAxes,
            fontsize=11, verticalalignment="top", fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="white"))

for _, r in summary.iterrows():
    ax.annotate(r["style_id"].split("_")[0], (r["tokens"], r["lat_mean"]),
                xytext=(8, 5), textcoords="offset points", fontsize=9, fontweight="bold")
ax.set_xlabel("Prompt size (~tokens)")
ax.set_ylabel("Mean latency (s)")
ax.set_title("Prompt Size vs Latency — Style Variants", fontweight="bold")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("prompt_style_results/fig_prompt_style_summary.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: prompt_style_results/fig_prompt_style_summary.png")


Saved: prompt_style_results/fig_prompt_style_summary.png


/var/folders/zy/jrpsl1v90xx47xwd7fws6np00000gn/T/ipykernel_28910/269880403.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


## Cell 7 — Per-style output inspection

In [16]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Per-style detail (chart types + sample titles)
# ═══════════════════════════════════════════════════════════════════

print(f"\n{'='*85}")
print("  PER-STYLE OUTPUT INSPECTION")
print(f"{'='*85}")

for sid in ORDER:
    sub = df[df["style_id"]==sid]
    if len(sub)==0: continue
    s_name = sub["style_name"].iloc[0]
    
    print(f"\n  ▸ {sid}: {s_name}  (~{int(sub['approx_tokens'].iloc[0])} tokens)")
    print(f"    Mean latency:  {sub['latency_s'].mean():.2f}s")
    print(f"    Mean quality:  {sub['quality_score'].mean():.2f}/8")
    print(f"    Annotations:   mean {sub['n_annotations'].mean():.1f}")
    print(f"    Data groups:   mean {sub['n_data_groups'].mean():.1f}")
    print(f"    Sample chart types:")
    for _, row in sub.iterrows():
        title = row["title_text"][:60] if isinstance(row["title_text"], str) else ""
        print(f"      Run {row['run']}: {row['charttype']:<12}  title=\"{title}\"")

print(f"\n{'='*85}")



  PER-STYLE OUTPUT INSPECTION

  ▸ S1_trigger_words: Trigger words only  (~10 tokens)
    Mean latency:  16.24s
    Mean quality:  7.33/8
    Annotations:   mean 1.7
    Data groups:   mean 4.0
    Sample chart types:
      Run 1: line          title="Teckentrup JVA 2021-2024 Umsatz monatlich"
      Run 2: line          title="Teckentrup JVA 2021-2024 Umsatz monatlich"
      Run 3: line          title="Teckentrup JVA: Umsatz 2021-2024 monatlich"

  ▸ S2_short_imperative: Short imperative  (~15 tokens)
    Mean latency:  21.91s
    Mean quality:  8.00/8
    Annotations:   mean 2.0
    Data groups:   mean 4.0
    Sample chart types:
      Run 1: line          title="Monthly Teckentrup JVA Revenue (2021-2024)"
      Run 2: line          title="Monthly Teckentrup JVA Revenue 2021-2024"
      Run 3: line          title="Monatlicher Teckentrup JVA Umsatz 2021-2024"

  ▸ S3_standard_german: Standard (current baseline)  (~34 tokens)
    Mean latency:  225.97s
    Mean quality:  8.00/8
    Ann

## Cell 8 — Thesis summary

In [17]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Thesis-ready summary
# ═══════════════════════════════════════════════════════════════════

print(f"\n{'='*75}")
print("  PROMPT STYLE BENCHMARK — THESIS SUMMARY")
print(f"{'='*75}\n")

# pick fastest and slowest
fastest = summary.nsmallest(1, "lat_mean").iloc[0]
slowest = summary.nlargest(1, "lat_mean").iloc[0]

print("  HEADLINE FINDINGS")
print("  -----------------")
print(f"   Fastest style: {fastest['style_name']:<25} → {fastest['lat_mean']:.1f}s, {fastest['qual_mean']:.1f}/8 quality")
print(f"   Slowest style: {slowest['style_name']:<25} → {slowest['lat_mean']:.1f}s, {slowest['qual_mean']:.1f}/8 quality")
print(f"   Speed ratio:   {slowest['lat_mean']/fastest['lat_mean']:.2f}x")
print()

# Quality preservation across styles
qual_min = summary["qual_mean"].min()
qual_max = summary["qual_mean"].max()
qual_preserved = qual_max - qual_min < 1.0
print(f"   Quality range: {qual_min:.1f} — {qual_max:.1f} /8")
print(f"   Quality preserved across styles: {'YES' if qual_preserved else 'NO'}")
print()

# Visual richness
trigger_annos = summary[summary["style_id"]=="S1_trigger_words"]["annos_mean"].iloc[0]
verbose_annos = summary[summary["style_id"]=="S5_verbose"]["annos_mean"].iloc[0]
print(f"   Annotations — trigger words: {trigger_annos:.1f}")
print(f"   Annotations — verbose:       {verbose_annos:.1f}")
print(f"   Verbose adds {verbose_annos-trigger_annos:.1f} more annotations on average")
print()

print("  THESIS IMPLICATIONS")
print("  -------------------")
if fastest["qual_mean"] >= 5 and trigger_annos >= verbose_annos * 0.5:
    print("   ✓ Trigger-word prompts are viable: latency reduced significantly")
    print("     while structural quality is maintained at acceptable levels.")
    print("     This means real-world fragmented business queries CAN be handled")
    print("     without rewriting them into full sentences.")
elif fastest["qual_mean"] >= 5 and trigger_annos < verbose_annos * 0.5:
    print("   ⚠  Trigger-word prompts trade visual richness for latency:")
    print(f"     fewer annotations ({trigger_annos:.1f} vs {verbose_annos:.1f}),")
    print("     similar structural quality but simpler output.")
    print("     This is a tradeoff for the user, not a free win.")
else:
    print("   ⚠  Trigger-word prompts degrade quality below the pass threshold.")
    print("     Verbose prompts are required for production-grade output.")

print(f"{'='*75}")



  PROMPT STYLE BENCHMARK — THESIS SUMMARY

  HEADLINE FINDINGS
  -----------------
   Fastest style: Trigger words only        → 16.2s, 7.3/8 quality
   Slowest style: Standard (current baseline) → 226.0s, 8.0/8 quality
   Speed ratio:   13.91x

   Quality range: 7.3 — 8.0 /8
   Quality preserved across styles: YES

   Annotations — trigger words: 1.7
   Annotations — verbose:       3.0
   Verbose adds 1.3 more annotations on average

  THESIS IMPLICATIONS
  -------------------
   ✓ Trigger-word prompts are viable: latency reduced significantly
     while structural quality is maintained at acceptable levels.
     This means real-world fragmented business queries CAN be handled
     without rewriting them into full sentences.
